# YOLOX Document Detection Training

Train YOLOX model for Algerian ID card detection.

**License:** Apache-2.0 (Free for commercial use)

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone YOLOX repository
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q -e .
!pip install -q onnx onnxruntime

## 2. Prepare Dataset

Create COCO format dataset for Algerian ID cards.

Expected structure:
```
dataset/
  ├── train/
  │   ├── images/
  │   └── annotations.json
  └── val/
      ├── images/
      └── annotations.json
```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy dataset from Drive
!mkdir -p /content/dataset
!cp -r /content/drive/MyDrive/datasets/algerian_id_cards/* /content/dataset/

## 3. Create Custom Config

In [ ]:
# Create custom config file
config_content = '''
#!/usr/bin/env python3
# -*- coding:utf-8 -*-

import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.depth = 0.33
        self.width = 0.50
        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]
        
        # Define model name and path
        self.output_dir = "/content/YOLOX/YOLOX_outputs"
        
        # ---------------- dataset config ---------------- #
        self.num_classes = 1  # Only ID card
        self.data_dir = "/content/dataset"
        self.train_ann = "train/annotations.json"
        self.val_ann = "val/annotations.json"
        
        # --------------- transform config ---------------- #
        self.mosaic_prob = 1.0
        self.mixup_prob = 0.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 10.0
        self.translate = 0.1
        self.mosaic_scale = (0.1, 2)
        self.enable_mixup = False
        
        # -------------- training config ----------------- #
        self.warmup_epochs = 5
        self.max_epoch = 100
        self.warmup_lr = 0
        self.basic_lr_per_img = 0.01 / 64.0
        self.scheduler = "yoloxwarmcos"
        self.no_aug_epochs = 15
        self.min_lr_ratio = 0.05
        self.ema = True
        self.weight_decay = 0.0005
        self.momentum = 0.9
        self.print_interval = 10
        self.eval_interval = 10
        self.save_history_ckpt = False
        
        # ----------------- dataloader config ----------------- #
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.no_aug = False
        self.degrees = 10.0
        self.translate = 0.1
        self.scale = (0.1, 2)
        self.mosaic_scale = (0.5, 1.5)
        self.shear = 2.0
        self.perspective = 0.0
        self.enable_mixup = True
        self.mixup_scale = (0.5, 1.5)
        self.mixup_prob = 0.5
        
    def get_model(self):
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
        import torch
        
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, torch.nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if getattr(self, "model", None) is None:
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels)
            self.model = YOLOX(backbone, head)

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model
    
    def get_data_loader(
        self, batch_size, is_distributed, no_aug=False, cache_img=False
    ):
        from yolox.data import (
            COCODataset,
            TrainTransform,
            YoloBatchSampler,
            DataLoader,
            InfiniteSampler,
            MosaicDetection,
        )
        import torch

        dataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=50,
                flip_prob=self.flip_prob,
                hsv_prob=self.hsv_prob,
            ),
            cache=cache_img,
        )

        dataset = MosaicDetection(
            dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=120,
                flip_prob=self.flip_prob,
                hsv_prob=self.hsv_prob,
            ),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            shear=self.shear,
            enable_mixup=self.enable_mixup,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob,
        )

        if is_distributed:
            batch_size = batch_size // torch.cuda.device_count()

        sampler = InfiniteSampler(len(dataset), seed=self.seed if self.seed else 0)

        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug,
        )

        dataloader_kwargs = {"num_workers": self.data_num_workers, "pin_memory": True}
        dataloader_kwargs["batch_sampler"] = batch_sampler

        train_loader = DataLoader(dataset, **dataloader_kwargs)

        return train_loader

    def get_eval_loader(self, batch_size, is_distributed, testdev=False):
        from yolox.data import COCODataset, ValTransform
        import torch

        valdataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name="val",
            img_size=self.test_size,
            preproc=ValTransform(legacy= False),
        )

        if is_distributed:
            batch_size = batch_size // torch.cuda.device_count()
            sampler = torch.utils.data.distributed.DistributedSampler(
                valdataset, shuffle=False
            )
        else:
            sampler = torch.utils.data.SequentialSampler(valdataset)

        dataloader_kwargs = {
            "num_workers": self.data_num_workers,
            "pin_memory": True,
            "sampler": sampler,
        }
        dataloader_kwargs["batch_size"] = batch_size
        val_loader = torch.utils.data.DataLoader(valdataset, **dataloader_kwargs)

        return val_loader

    def get_evaluator(self, batch_size, is_distributed, testdev=False):
        from yolox.evaluators import COCOEvaluator

        val_loader = self.get_eval_loader(batch_size, is_distributed, testdev)
        evaluator = COCOEvaluator(
            dataloader=val_loader,
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )
        return evaluator
'''

with open('/content/YOLOX/exps/custom/yolox_idcard.py', 'w') as f:
    f.write(config_content)
    
print('Config file created!')

## 4. Train Model

In [ ]:
# Download pretrained weights (optional)
!wget https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth
!mkdir -p /content/YOLOX/pretrained
!mv yolox_s.pth /content/YOLOX/pretrained/

In [ ]:
# Start training
!python tools/train.py \
    -f exps/custom/yolox_idcard.py \
    -d 1 \
    -b 16 \
    --fp16 \
    -o \
    -c pretrained/yolox_s.pth

## 5. Export to ONNX

In [ ]:
# Export trained model to ONNX
!python tools/export_onnx.py \
    --output-name yolox_idcard.onnx \
    -f exps/custom/yolox_idcard.py \
    -c YOLOX_outputs/yolox_idcard/best_ckpt.pth

## 6. Test Model

In [ ]:
# Test inference
from PIL import Image
import matplotlib.pyplot as plt

# Load test image
test_image = '/content/dataset/val/images/test_001.jpg'
img = Image.open(test_image)

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.title('Test Image')
plt.axis('off')
plt.show()

In [ ]:
# Run inference
!python tools/demo.py \
    image \
    -f exps/custom/yolox_idcard.py \
    -c YOLOX_outputs/yolox_idcard/best_ckpt.pth \
    --path /content/dataset/val/images/test_001.jpg \
    --conf 0.5 \
    --nms 0.45 \
    --tsize 640 \
    --save_result

## 7. Save Model to Drive

In [ ]:
# Copy ONNX model to Drive
!cp yolox_idcard.onnx /content/drive/MyDrive/models/
!cp YOLOX_outputs/yolox_idcard/best_ckpt.pth /content/drive/MyDrive/models/

print('Model saved to Google Drive!')

## Validation Metrics

Expected results:
- mAP@0.5: > 0.95
- Inference time: < 10ms on GPU
- Model size: ~30MB (YOLOX-S)